# 4.14 · 分位数回归 / Quantile Regression

> **课程定位 / Where this fits**
> **Part 4 第 14 课**。前 13 课都预测**条件均值** $\mathbb{E}[y|\mathbf{x}]$。但很多场景要的是**条件分位数**: "95% 情况下配送时间不超过多少?" "房价的 10%-90% 区间?" 分位数回归用 **pinball loss** 直接预测任意分位数, 并天然给出**预测区间**。
> Beyond the conditional mean: predict conditional quantiles via pinball loss, yielding prediction intervals.

> 💡 **面试相关 / Interview-relevant**
> - "怎么给回归预测一个区间而非点" ★★★★（分位数回归）
> - "pinball loss 是什么" ★★★
> - "分位数回归 vs OLS 区别" ★★★★（中位数 vs 均值, 稳健）
> - "预测区间 vs 置信区间" ★★★（2.5 接口）

---

## 学习目标 / Learning Objectives
1. 理解为什么"只预测均值"不够（异方差/风险/SLA 场景）。
2. 掌握 **pinball (分位数) loss** 及其几何。
3. 用 sklearn 的 `QuantileRegressor` 和 GBDT quantile loss 预测多个分位数。
4. 用分位数构造**预测区间**（接 2.5 的预测区间概念）。
5. 理解 MAE = 50% 分位数回归（中位数）。

## 目录 / TOC
1. [为什么需要分位数 ⭐](#1)
2. [pinball loss ⭐](#2)
3. [数据: 异方差](#3)
4. [线性分位数回归 + 预测区间](#4)
5. [GBDT 分位数回归(非线性)](#5)
6. [区间覆盖率检验 (2.5 接口)](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 为什么需要分位数 ⭐ / Why Quantiles

均值预测 $\hat{y} = \mathbb{E}[y|\mathbf{x}]$ 回答"平均会怎样", 但很多决策要的是**不确定性的边界**：

| 场景 | 要的不是均值, 而是 |
|---|---|
| 配送 SLA | "**95%** 订单在多少分钟内送达"(P95) |
| 库存 | "**90%** 概率不缺货需备多少货"(高分位) |
| 风险/VaR | "**最坏 5%** 情况亏多少"(低分位, 2.12) |
| 房价区间 | "这房子价格 **10%-90% 区间**" |

而且 OLS 假设**同方差**——但真实数据常**异方差**（4.2: 收入越高波动越大）。**均值线无法表达"波动随 x 变化"**, 分位数回归能: 不同分位数的线**会张开/收窄**, 直接显示条件分布的形状。
The mean can't express varying spread; quantile lines fan out under heteroscedasticity.


<a id="2"></a>
## 2. pinball loss ⭐ / The Pinball Loss

预测 $\tau$ 分位数（如 $\tau=0.9$）, 用**非对称的 pinball loss**（也叫分位数损失/检验损失）：

$$L_\tau(y, \hat{y}) = \begin{cases} \tau\,(y - \hat{y}) & \text{if } y \ge \hat{y} \;(\text{低估})\\ (1-\tau)\,(\hat{y} - y) & \text{if } y < \hat{y}\;(\text{高估}) \end{cases}$$

**关键 = 非对称惩罚**:
- $\tau=0.9$: 低估（$\hat{y}<y$）惩罚 $0.9$, 高估只惩罚 $0.1$ → 模型被推着**往高预测** → 学到高分位
- $\tau=0.5$: 对称 ($0.5/0.5$) → 最小化 = **绝对误差 = 中位数**（4.15 稳健回归的接口！）
- $\tau=0.1$: 高估重罚 → 往低预测 → 低分位

**为什么最小化 pinball loss 得到 $\tau$ 分位数**: 在最优点, "低估的概率 × τ = 高估的概率 × (1-τ)" → $\Pr(y < \hat{y}) = \tau$, 正是分位数定义。
Minimizing pinball loss yields the τ-quantile because the asymmetric penalty balances at P(y<ŷ)=τ.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 画 pinball loss 几何 / pinball loss geometry
err = np.linspace(-3, 3, 300)   # err = y - ŷ
fig, ax = plt.subplots(figsize=(7, 4))
for tau, c in [(0.1, "C0"), (0.5, "C1"), (0.9, "C2")]:
    loss = np.where(err >= 0, tau*err, (tau-1)*err)
    ax.plot(err, loss, color=c, lw=2, label=f"τ={tau}")
ax.axvline(0, color="k", lw=0.5); ax.legend()
ax.set_xlabel("误差 (y - ŷ)"); ax.set_ylabel("pinball loss")
ax.set_title("Pinball loss: 非对称 V 形\nτ=0.9 时低估(右)惩罚陡, τ=0.5 对称=MAE")
plt.tight_layout(); plt.show()
print("τ=0.9: 右侧(低估)斜率陡(0.9), 左侧(高估)平(0.1) → 推模型往高预测 = 高分位")
print("τ=0.5: 两侧对称 → 就是绝对误差 MAE → 最优解是中位数")


<a id="3"></a>
## 3. 数据: 异方差 / Heteroscedastic Data

构造一个**方差随 x 增大**的数据集——均值线无法描述它, 分位数线才能。


In [ ]:
# 异方差: 噪声随 x 增大 / heteroscedastic: noise grows with x
n = 1000
x = np.sort(rng.uniform(0, 10, n))
y = 2*x + rng.normal(0, 0.5 + 0.6*x, n)    # 噪声 std 随 x 线性增大!
X = x.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.2, s=10)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("异方差数据: 散布随 x 张开 (喇叭形)\n均值线无法表达'波动越来越大'")
plt.tight_layout(); plt.show()
print("数据呈喇叭形 (4.2 异方差): x 小时点集中, x 大时散开")
print("单一均值线只给中心趋势, 分位数回归能描述'张开的宽度'")


<a id="4"></a>
## 4. 线性分位数回归 + 预测区间 / Linear Quantile Regression


In [ ]:
from sklearn.linear_model import QuantileRegressor, LinearRegression

x_plot = np.linspace(0, 10, 100).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=8)

# OLS 均值线 / mean line
ols = LinearRegression().fit(X, y)
ax.plot(x_plot, ols.predict(x_plot), "k--", lw=2, label="OLS 均值")

# 多个分位数 / multiple quantiles
quantiles = {0.1: "C0", 0.5: "C1", 0.9: "C2"}
preds = {}
for q, c in quantiles.items():
    qr = QuantileRegressor(quantile=q, alpha=0, solver="highs").fit(X, y)
    preds[q] = qr.predict(x_plot)
    ax.plot(x_plot, preds[q], color=c, lw=2, label=f"分位数 τ={q}")
# 填充 10-90% 预测区间 / shade the prediction interval
ax.fill_between(x_plot.ravel(), preds[0.1], preds[0.9], alpha=0.12, color="green",
                label="80% 预测区间 (P10-P90)")
ax.legend(fontsize=9); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("分位数回归: 分位数线张开 = 捕捉异方差; P10-P90 = 80% 预测区间")
plt.tight_layout(); plt.show()
print("三条分位数线随 x 张开 (而 OLS 只有一条均值线) → 直接显示波动随 x 增大")
print("P10 到 P90 之间 = 80% 预测区间, 比单点预测信息丰富得多")


**预测区间 vs 置信区间**（2.5 节的区分, 这里再强调）：
- **置信区间 (CI)**: 关于**参数/均值**的不确定性（如"回归线在 x=5 处的均值在哪")——窄
- **预测区间 (PI)**: 关于**单个新观测 y**落在哪——**宽得多**（包含个体噪声）

分位数回归的 P10-P90 给的是**预测区间**（单个房子的价格范围）, 不是均值的置信区间。这是实际决策（"这单配送会不会超时"）真正需要的。
Quantile regression gives a prediction interval (where a single new y falls), much wider than a confidence interval on the mean.


<a id="5"></a>
## 5. GBDT 分位数回归(非线性) / GBDT Quantile

4.13 说过 boosting **换损失就能做任何事**。GBDT 用 pinball loss → **非线性分位数回归**。sklearn 的 `GradientBoostingRegressor(loss="quantile", alpha=τ)`。


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# 非线性异方差数据 / nonlinear heteroscedastic
y_nl = np.sin(x) * x + rng.normal(0, 0.3 + 0.5*x, n)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y_nl, alpha=0.15, s=8)
gbdt_preds = {}
for q, c in [(0.1,"C0"),(0.5,"C1"),(0.9,"C2")]:
    g = GradientBoostingRegressor(loss="quantile", alpha=q, n_estimators=100,
                                  max_depth=3, learning_rate=0.1, random_state=0).fit(X, y_nl)
    gbdt_preds[q] = g.predict(x_plot)
    ax.plot(x_plot, gbdt_preds[q], color=c, lw=2, label=f"GBDT τ={q}")
ax.fill_between(x_plot.ravel(), gbdt_preds[0.1], gbdt_preds[0.9], alpha=0.12, color="green")
ax.legend(); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("GBDT 分位数回归: 非线性 + 异方差 都能捕捉")
plt.tight_layout(); plt.show()
print("GBDT 分位数同时捕捉非线性曲线 + 张开的区间 — boosting 换损失的威力(4.13)")
print("💡 也可用 XGBoost/LightGBM 的 quantile objective, 或 RF 的 quantile-forest 扩展")


<a id="6"></a>
## 6. 区间覆盖率检验 / Interval Coverage Check

**怎么验证分位数预测准不准**？用 2.5 节的金标准——**覆盖率**: P10-P90 的 80% 预测区间, 应该真的罩住约 80% 的测试点。


In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
lo = QuantileRegressor(quantile=0.1, alpha=0, solver="highs").fit(X_tr, y_tr).predict(X_te)
hi = QuantileRegressor(quantile=0.9, alpha=0, solver="highs").fit(X_tr, y_tr).predict(X_te)

coverage = np.mean((y_te >= lo) & (y_te <= hi))
print(f"P10-P90 名义 80% 预测区间:")
print(f"  实际覆盖率 = {coverage:.1%}  (目标 80%)")
print(f"  → 接近 80% = 分位数预测校准良好 (2.5 覆盖率验证)")

# 各分位数的 pinball loss (验证模型确实在最小化它) / pinball loss check
from sklearn.metrics import mean_pinball_loss
for q in [0.1, 0.5, 0.9]:
    qr = QuantileRegressor(quantile=q, alpha=0, solver="highs").fit(X_tr, y_tr)
    pl = mean_pinball_loss(y_te, qr.predict(X_te), alpha=q)
    print(f"  τ={q}: test pinball loss = {pl:.3f}")


<a id="7"></a>
## 7. 小结 / Summary

```
分位数回归: 预测条件分位数 (非均值) → 异方差/风险/SLA/预测区间
pinball loss ⭐: 非对称 V 形, τ=0.9 低估重罚→高预测, τ=0.5=MAE=中位数
  最优点满足 P(y<ŷ)=τ = 分位数定义
线性: QuantileRegressor; 非线性: GBDT(loss='quantile') — boosting 换损失(4.13)
预测区间(P10-P90, 含个体噪声, 宽) ≠ 置信区间(均值不确定性, 窄)
验证: 覆盖率 (80% 区间应罩住~80% 点, 2.5)
```

### 💡 面试速查
1. **要区间不要点 → 分位数回归** (预测多个分位数)
2. **pinball loss**: 非对称惩罚, τ 决定往高/低预测
3. **τ=0.5 = MAE = 中位数回归** (稳健, 接 4.15)
4. **预测区间 ≠ 置信区间**: PI 含个体噪声更宽
5. **GBDT 换 pinball loss** = 非线性分位数 (boosting 通用性)

### 下一节
**4.15 稳健回归**——τ=0.5 分位数(中位数)已经比均值抗异常值。稳健回归专门对付异常值: Huber / RANSAC / Theil-Sen。
